# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

_All data entities (record sets, fields, columns, etc.) are referenced by their `@id` as required for FAIR interoperability and reproducibility._

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and browse record sets using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Preview dataset metadata
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}\n")
print(f"License: {dataset.metadata.license}\n")

## 2. Data Overview

Review available record sets, their `@id`s, and available fields. All subsequent code will reference each data entity by its `@id`.

In [ ]:
# List all record sets by their `@id`
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print(f"  Fields in this RecordSet:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()


### Explore a Sample Record
Let's print the first record for each RecordSet by `@id` as an example.

In [ ]:
for rs in record_sets:
    print(f"First record for RecordSet '{rs.name}' (@id: {rs.id}):")
    records = dataset.records(record_set=rs.id)
    try:
        print(next(records))
    except StopIteration:
        print("  No records found.")
    print("-")

## 3. Data Extraction

Let's extract the main quantitative data set as a DataFrame for processing and analysis. All entities are referenced by their Croissant `@id`.

**Choose the main record set**: For this dataset, the main record set typically contains protein quantifications. We'll use the first tabular record set found (commonly has "protein" in name or similar) as an example. Adjust the variable below if exploring with another RecordSet.

In [ ]:
# Select the main data RecordSet (adjust if desired)
# For this example we pick the first record set with a tabular structure.

protein_record_set_id = None
for rs in record_sets:
    # Heuristic: look for the main protein quant data
    if 'protein' in rs.name.lower() or 'quant' in rs.name.lower():
        protein_record_set_id = rs.id
        break
# Default fallback to first record set
if protein_record_set_id is None and len(record_sets) > 0:
    protein_record_set_id = record_sets[0].id

print(f"Using RecordSet @id: {protein_record_set_id}")

# List all record sets (for documentation)
all_record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in all_record_set_ids:
    data = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(data)
    dataframes[rs_id] = df
    print(f"RecordSet @id: {rs_id}, columns: {df.columns.tolist()}")

# Preview main dataset
df_main = dataframes[protein_record_set_id]
df_main.head()

## 4. Exploratory Data Analysis (EDA)

Apply basic filtering and feature processing steps:
- Filter records based on a protein abundance threshold.
- Normalize a numeric column (e.g., peptide count, coverage, abundance).
- Group by a categorical field (if relevant, e.g., sample or category).

#### Find Fields for EDA
Browse numeric and categorical columns and select the correct Croissant field `@id` for each operation.

In [ ]:
# List all columns for the main record set
print("Columns in the main DataFrame:")
print(df_main.columns.tolist())

# For demonstration, pick numeric_field (e.g. 'abundance', 'peptide_count', or similar)
# and a group field (e.g. 'sample', 'accession', or similar)
numeric_field = None
possible_numeric_fields = ['abundance', 'intensity', 'peptide_count', 'coverage', 'MW', 'pI']
for col in df_main.columns:
    if any(field in col.lower() for field in possible_numeric_fields):
        numeric_field = col
        break
if numeric_field is None:
    print("No matching numeric field found, using first numeric-looking column.")
    for col in df_main.columns:
        if pd.api.types.is_numeric_dtype(df_main[col]):
            numeric_field = col
            break
print(f"Numeric field selected: {numeric_field}")

group_field = None
possible_group_fields = ['sample', 'accession', 'modification']
for col in df_main.columns:
    if any(field in col.lower() for field in possible_group_fields):
        group_field = col
        break
print(f"Group field selected: {group_field}")

if numeric_field is not None:
    # Remove missing/NaN rows for target field
    filtered = df_main[df_main[numeric_field].notna()]
    # Convert column to numeric (errors coerced to NaN, which are dropped)
    filtered[numeric_field] = pd.to_numeric(filtered[numeric_field], errors='coerce')
    threshold = filtered[numeric_field].quantile(0.8)  # Top 20%
    filtered_df = filtered[filtered[numeric_field] >= threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field} >= {threshold:.2f}")
    # Normalize
    norm_name = f"{numeric_field}_normalized"
    filtered_df[norm_name] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, norm_name]].head())
    # Grouping
    if group_field is not None and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"\nGrouped mean {numeric_field} by {group_field} (top results):")
        print(grouped.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the normalized numeric field and the top protein or group by abundance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and len(filtered_df) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[norm_name], bins=30, kde=True)
    plt.title(f"Distribution of normalized {numeric_field}")
    plt.xlabel(norm_name)
    plt.ylabel("Count")
    plt.show()
    # Top proteins or groups
    if group_field is not None and group_field in filtered_df.columns:
        top_n = 10
        top_groups = filtered_df.groupby(group_field)[numeric_field].mean().nlargest(top_n)
        plt.figure(figsize=(10,5))
        sns.barplot(x=top_groups.index, y=top_groups.values)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Top {top_n} {group_field}s by mean {numeric_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No suitable field found for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR^2 mass spectrometry dataset using `mlcroissant`.
- Identified all available record sets and explored their fields using `@id`.
- Extracted the main protein quantification record set as a DataFrame for analysis.
- Performed basic exploratory data analysis including filtering, normalization, and grouping by field.
- Visualized the distribution and top categories of protein abundance.

**Tip:** You can extend this analysis by referencing specific `@id`s for fields of interest, or exploring additional record sets as described in the Croissant schema.

_All data selection and analysis were performed with Croissant-compliant references for reproducibility._